In [2]:
"""
依赖诱导恶化状态转移模型演示
"""

import sys
import os
from pathlib import Path

# 获取当前 notebook 所在的目录（Jupyter 中）
try:
    # 尝试获取 __file__（在 .py 文件中有效）
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 在 Jupyter notebook 中，使用当前工作目录
    current_dir = os.getcwd()
    
# 将当前目录添加到系统路径
sys.path.append(current_dir)

from models.dependency_model import DependencyDeteriorationModel
from models.base_model import EventType


In [3]:
def demo_dependency_model():
    """演示依赖诱导恶化状态转移模型"""
    print("=" * 80)
    print("依赖诱导恶化状态转移模型演示")
    print("=" * 80)
    
    # 创建模型实例 - n=3
    model = DependencyDeteriorationModel(n=3)
    
    n_val = model.model_params['n']
    
    print(f"\n📊 模型信息:")
    print(f"   参数: n={n_val}")
    print(f"   初始状态: (0,0,1) 不健康")
    print(f"   可用干预:")
    print(f"      - 干预: (1,0,1) 🔵")
    print(f"      - 无干预: (0,0,0) ⚪")
    
    print(f"\n📖 状态说明:")
    print(f"   (0,0,1) → 不健康状态")
    print(f"   (0,0,0) → 稳定健康状态")
    print(f"   (1,0,1) → 恶化状态")
    
    print(f"\n📖 事件说明:")
    print(f"   E5a (稳定健康): 最近 {n_val} 步都是干预 → 健康 (0,0,0)")
    print(f"   E5b (恶化): 连续 {n_val} 次干预后停止 → 恶化 (1,0,1)")
    print(f"   E5c (无有效转移): 其他情况 → 不健康 (0,0,1)")
    
    # 场景1: 稳定健康 (连续n次干预)
    print("\n" + "=" * 80)
    print(f"场景1: 稳定健康 - 连续 {n_val} 次干预")
    print("=" * 80)
    
    interventions_scene1 = [(1, 0, 1)] * n_val
    
    result1 = model.simulate(interventions_scene1, return_details=True)
    _print_results(result1, model, title=f"连续{n_val}次干预 → 稳定健康")
    
    # 场景2: 恶化 (连续n次干预后停止)
    print("\n" + "=" * 80)
    print(f"场景2: 恶化 - 连续 {n_val} 次干预后停止")
    print("=" * 80)
    
    interventions_scene2 = [(1, 0, 1)] * n_val + [(0, 0, 0)]
    
    result2 = model.simulate(interventions_scene2, return_details=True)
    _print_results(result2, model, title=f"连续{n_val}次干预后停止 → 恶化")
    
    # 场景3: 恶化后重新恢复
    print("\n" + "=" * 80)
    print(f"场景3: 恶化后重新恢复 - 恶化后再连续 {n_val} 次干预")
    print("=" * 80)
    
    interventions_scene3 = (
        [(1, 0, 1)] * n_val +   # 达到健康
        [(0, 0, 0)] +           # 停止干预，导致恶化
        [(1, 0, 1)] * n_val     # 重新干预，恢复健康
    )
    
    result3 = model.simulate(interventions_scene3, return_details=True)
    _print_results(result3, model, title="恶化后重新干预 → 恢复健康")
    
    # 场景4: 未达到阈值
    print("\n" + "=" * 80)
    print(f"场景4: 未达到阈值 - 连续 {n_val-1} 次干预")
    print("=" * 80)
    
    interventions_scene4 = [(1, 0, 1)] * (n_val - 1)
    
    result4 = model.simulate(interventions_scene4, return_details=True)
    _print_results(result4, model, title=f"连续{n_val-1}次干预（未达到{n_val}次）→ 不健康")
    
    # 场景5: 部分干预后停止（无恶化）
    print("\n" + "=" * 80)
    print(f"场景5: 部分干预后停止 - 连续 {n_val-1} 次干预后停止")
    print("=" * 80)
    
    interventions_scene5 = [(1, 0, 1)] * (n_val - 1) + [(0, 0, 0)]
    
    result5 = model.simulate(interventions_scene5, return_details=True)
    _print_results(result5, model, title=f"连续{n_val-1}次干预后停止 → 不健康（未恶化）")


def demo_dependency_cycle():
    """演示依赖循环：健康 → 恶化 → 恢复 → 恶化"""
    print("\n" + "=" * 80)
    print("依赖循环演示：健康 ↔ 恶化")
    print("=" * 80)
    
    model = DependencyDeteriorationModel(n=3)
    n_val = model.model_params['n']
    
    # 创建完整的依赖循环
    interventions = []
    for _ in range(3):  # 3个循环
        interventions.extend([(1, 0, 1)] * n_val)  # 恢复到健康
        interventions.append((0, 0, 0))            # 停止干预，导致恶化
    
    result = model.simulate(interventions, return_details=True)
    
    print(f"\n干预序列（共 {len(interventions)} 步）:")
    for t, inter in enumerate(interventions, 1):
        if inter == model.INTERVENTION:
            print(f"   t={t:2d}: {inter} 🔵 干预")
        else:
            print(f"   t={t:2d}: {inter} ⚪ 无干预")
    
    print(f"\n状态演变:")
    print("-" * 80)
    print(f"{'步数':<6} {'可观测状态':<20} {'状态说明':<15} {'事件':<40}")
    print("-" * 80)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        
        if state == model.HEALTHY:
            state_name = "健康"
        elif state == model.DETERIORATED:
            state_name = "恶化"
        else:
            state_name = "不健康"
        
        if i == 0:
            event_str = "-"
        else:
            event = result["events"][i-1]
            if event == EventType.COMPLETE_TRANSITION:
                event_str = "E5a (稳定健康) ⭐"
            elif event == EventType.DETERIORATION:
                event_str = "E5b (恶化) 💀"
            else:
                event_str = "E5c (无有效转移)"
        
        display_state = f"{state} ({state_name})"
        print(f"{i:<6} {display_state:<20} {state_name:<15} {event_str:<40}")
    
    print(f"\n💡 观察:")
    print(f"   - 连续{n_val}次干预后，状态变为健康")
    print(f"   - 停止干预后，状态立即恶化为 (1,0,1)")
    print(f"   - 恶化后需要重新连续{n_val}次干预才能恢复")


def demo_parameter_comparison():
    """参数对比演示"""
    print("\n" + "=" * 80)
    print("参数对比：不同 n 值的影响")
    print("=" * 80)
    
    param_sets = [
        {"n": 2, "name": "n=2 (快速恢复)"},
        {"n": 3, "name": "n=3 (中等恢复)"},
        {"n": 5, "name": "n=5 (慢速恢复)"},
    ]
    
    for params in param_sets:
        model = DependencyDeteriorationModel(n=params["n"])
        n_val = model.model_params['n']
        
        print(f"\n{params['name']}:")
        print(f"   需要 {n_val} 次连续干预才能达到健康")
        print(f"   恶化后需要重新 {n_val} 次连续干预才能恢复")
        
        # 测试恢复时间
        interventions = [(1, 0, 1)] * n_val + [(0, 0, 0)] + [(1, 0, 1)] * n_val
        result = model.simulate(interventions)
        
        # 找到第一次达到健康的位置
        healthy_steps = [i for i, s in enumerate(result["states"]) if s == model.HEALTHY]
        if len(healthy_steps) >= 2:
            print(f"   第一次恢复: 第 {healthy_steps[0]} 步")
            print(f"   第二次恢复: 第 {healthy_steps[1]} 步")


def _print_results(result: dict, model, title: str = ""):
    """打印模拟结果"""
    if title:
        print(f"\n{title}")
    
    print(f"\n干预序列:")
    for t, inter in enumerate(result["interventions"], 1):
        if inter == model.INTERVENTION:
            print(f"   t={t:2d}: {inter} 🔵 干预")
        else:
            print(f"   t={t:2d}: {inter} ⚪ 无干预")
    
    print(f"\n状态演变:")
    print("-" * 80)
    print(f"{'步数':<6} {'可观测状态':<20} {'状态说明':<15} {'事件':<40}")
    print("-" * 80)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        
        if state == model.HEALTHY:
            state_name = "健康"
        elif state == model.DETERIORATED:
            state_name = "恶化"
        else:
            state_name = "不健康"
        
        if i == 0:
            event_str = "-"
        else:
            event = result["events"][i-1]
            if event == EventType.COMPLETE_TRANSITION:
                event_str = "E5a (稳定健康) ⭐"
            elif event == EventType.DETERIORATION:
                event_str = "E5b (恶化) 💀"
            else:
                event_str = "E5c (无有效转移)"
        
        display_state = f"{state} ({state_name})"
        print(f"{i:<6} {display_state:<20} {state_name:<15} {event_str:<40}")
    
    # 显示内部状态
    internal = model.get_internal_state()
    if internal["is_healthy"]:
        print(f"\n💡 当前处于稳定健康状态")
    if internal["is_deteriorated"]:
        print(f"\n⚠️  当前处于恶化状态，需要干预才能恢复")


In [4]:
if __name__ == "__main__":
    demo_dependency_model()
    demo_dependency_cycle()
    demo_parameter_comparison()

依赖诱导恶化状态转移模型演示

📊 模型信息:
   参数: n=3
   初始状态: (0,0,1) 不健康
   可用干预:
      - 干预: (1,0,1) 🔵
      - 无干预: (0,0,0) ⚪

📖 状态说明:
   (0,0,1) → 不健康状态
   (0,0,0) → 稳定健康状态
   (1,0,1) → 恶化状态

📖 事件说明:
   E5a (稳定健康): 最近 3 步都是干预 → 健康 (0,0,0)
   E5b (恶化): 连续 3 次干预后停止 → 恶化 (1,0,1)
   E5c (无有效转移): 其他情况 → 不健康 (0,0,1)

场景1: 稳定健康 - 连续 3 次干预

连续3次干预 → 稳定健康

干预序列:
   t= 1: (1, 0, 1) 🔵 干预
   t= 2: (1, 0, 1) 🔵 干预
   t= 3: (1, 0, 1) 🔵 干预

状态演变:
--------------------------------------------------------------------------------
步数     可观测状态                状态说明            事件                                      
--------------------------------------------------------------------------------
0      (0, 0, 1) (不健康)      不健康             -                                       
1      (0, 0, 1) (不健康)      不健康             E5c (无有效转移)                             
2      (0, 0, 1) (不健康)      不健康             E5c (无有效转移)                             
3      (0, 0, 0) (健康)       健康              E5a (稳定健康) ⭐                      